Importation des bibliothèques

In [159]:
import spotipy
from spotipy.oauth2 import SpotifyOAuth
from config import SPOTIPY_CLIENT_ID, SPOTIPY_CLIENT_SECRET, SPOTIPY_REDIRECT_URI
import requests
import pandas as pd
from IPython.display import display
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

Initialiser le client Spotify

In [160]:
# 1. Obtenir un token (Client Credentials Flow)
def get_spotify_token():
    auth_url = "https://accounts.spotify.com/api/token"
    auth_response = requests.post(
        auth_url,
        data={"grant_type": "client_credentials"},
        auth=(SPOTIPY_CLIENT_ID, SPOTIPY_CLIENT_SECRET)
    )
    auth_response.raise_for_status()
    return auth_response.json()["access_token"]
# Portées (scopes) nécessaires pour récupérer tes morceaux


SCOPE = [
    "user-read-recently-played",  # Pour récupérer ton historique récent
    "user-top-read"                # Pour récupérer tes morceaux préférés
]

# Initialise le client Spotify
sp = spotipy.Spotify(auth_manager=SpotifyOAuth(
    client_id=SPOTIPY_CLIENT_ID,
    client_secret=SPOTIPY_CLIENT_SECRET,
    redirect_uri=SPOTIPY_REDIRECT_URI,
    scope=SCOPE
))

In [161]:
# 1. Obtenir un token (Client Credentials Flow)
def get_spotify_token():
    auth_url = "https://accounts.spotify.com/api/token"
    auth_response = requests.post(
        auth_url,
        data={"grant_type": "client_credentials"},
        auth=(SPOTIPY_CLIENT_ID, SPOTIPY_CLIENT_SECRET)
    )
    auth_response.raise_for_status()
    return auth_response.json()["access_token"]
# Portées (scopes) nécessaires pour récupérer tes morceaux


SCOPE = [
    "user-read-recently-played",  # Pour récupérer ton historique récent
    "user-top-read"                # Pour récupérer tes morceaux préférés
]

# Initialise le client Spotify
sp = spotipy.Spotify(auth_manager=SpotifyOAuth(
    client_id=SPOTIPY_CLIENT_ID,
    client_secret=SPOTIPY_CLIENT_SECRET,
    redirect_uri=SPOTIPY_REDIRECT_URI,
    scope=SCOPE
))

récupérer les 10 dernièes chansons écoutées avec l'API web de Spotify

In [162]:
# Récupère tes 10 derniers morceaux écoutés
recent_tracks = sp.current_user_recently_played(limit=10)

print("🎵 Tes 10 derniers morceaux écoutés :")
for idx, item in enumerate(recent_tracks['items'], 1):
    track = item['track']
    artists = ', '.join([artist['name'] for artist in track['artists']])
    played_at = item['played_at']  # Date et heure d'écoute
    print(f"{idx}. {track['name']} - {artists} (ID: {track['id']}) | Écoute le {played_at}")

🎵 Tes 10 derniers morceaux écoutés :
1. Un monde à l'autre - GP Explorer, GIMS, La Mano 1.9, SCH (ID: 3XHcbhTEqJ21TTkQFUe6bG) | Écoute le 2026-09-30T09:54:19.114Z
2. Waka Waka (This Time for Africa) [The Official 2010 FIFA World Cup (TM) Song] (feat. Freshlyground) - Shakira, Freshlyground (ID: 4B0ScPwmG3oAOg9Ek7ODdm) | Écoute le 2026-09-29T20:23:32.085Z
3. Ma mélodie (feat. GIMS) - Dry, GIMS (ID: 0BU5e8rDml803bp7mbouC0) | Écoute le 2026-09-29T19:56:51.598Z
4. Avant la douane - Jul (ID: 3AwXhQWatI5b1sLIhsnOdP) | Écoute le 2026-09-29T19:52:19.326Z
5. JCVD - Jul (ID: 10iesoGb4mCYTcur1QfdO9) | Écoute le 2026-09-29T19:48:34.547Z
6. Comme d'hab - Bigflo & Oli (ID: 6BDMEFAXWg3yOjVaZW0KrL) | Écoute le 2026-09-29T19:44:58.210Z
7. Warano-Style - GIMS (ID: 7JbaSH5ZvdmKHxsjPXfPCk) | Écoute le 2026-09-29T19:38:51.072Z
8. And We Danced (feat. Ziggy Stardust) - Macklemore (ID: 1FCQEg7wOK9IIBuxx63krr) | Écoute le 2026-09-29T19:33:17.328Z
9. Without You (feat. Usher) - Radio Edit - David Guetta, USHER

Aller chercher les audio features de toutes les chansons grâce à l'API Reccobeats

In [163]:
# API reccobeats to have the audio features
RECCO_URL = "https://api.reccobeats.com/v1/audio-features"
#function to get audio features
def get_audio_features(spotify_ids):
    """Récupère les audio features via ReccoBeats à partir d'IDs Spotify."""
    response = requests.get(
        RECCO_URL,
        params={"ids": ",".join(spotify_ids)},
        headers={"Accept": "application/json"},
        timeout=10,
    )
    response.raise_for_status()
    data = response.json()
    # La réponse est généralement de la forme {"content": [ ... ]}
    return data.get("content", data) if isinstance(data, dict) else data


track_ids = [item["track"]["id"] for item in recent_tracks["items"]]
track_names = {item["track"]["id"]: item["track"]["name"] for item in recent_tracks["items"]}

# get audio features of track_ids
features = get_audio_features(track_ids)

Créer une matrice contenant les audio features de toutes les chansons

In [164]:
#create a dataframe of features
features_df = pd.DataFrame(features)
matrix = features_df.iloc[1:11, 3:14]

feature_columns = ["acousticness", "danceability", "energy", "instrumentalness",
                    "key", "liveness", "loudness", "mode", "speechiness", "tempo", "valence"]
features_matrix = matrix[feature_columns].values  # Matrice numpy des features

print("Matrice des features :")
print(features_matrix)

Matrice des features :
[[ 6.73000e-01  8.44000e-01  6.51000e-01  7.08000e-05  0.00000e+00
   1.74000e-01 -9.16200e+00  0.00000e+00  2.30000e-01  1.38038e+02
   7.95000e-01]
 [ 2.58000e-01  8.61000e-01  6.98000e-01  0.00000e+00  7.00000e+00
   5.72000e-02 -5.09800e+00  1.00000e+00  2.91000e-01  9.20490e+01
   7.52000e-01]
 [ 2.17000e-01  9.01000e-01  9.03000e-01  1.04000e-03  1.00000e+00
   1.18000e-01 -4.20000e+00  0.00000e+00  2.47000e-01  9.30000e+01
   7.90000e-01]
 [ 4.07000e-02  8.23000e-01  9.66000e-01  1.85000e-01  5.00000e+00
   2.56000e-01 -4.62000e+00  0.00000e+00  9.80000e-02  1.28033e+02
   5.90000e-01]
 [ 3.01000e-03  7.41000e-01  7.95000e-01  0.00000e+00  0.00000e+00
   9.08000e-02 -5.31500e+00  1.00000e+00  1.09000e-01  9.20140e+01
   3.34000e-01]
 [ 6.81000e-03  7.00000e-01  8.98000e-01  3.98000e-05  2.00000e+00
   1.58000e-01 -5.45600e+00  0.00000e+00  1.54000e-01  1.20038e+02
   2.34000e-01]
 [ 4.56000e-01  7.58000e-01  6.77000e-01  0.00000e+00  5.00000e+00
   9.54000

Normaliser la matrice de features

In [165]:
# Initialiser le scaler
scaler = MinMaxScaler()

# Normaliser la matrice
normalized_features = scaler.fit_transform(features_matrix)

print("Matrice normalisée :\n", normalized_features)


# Calculer la matrice de similarité cosinus
similarity_matrix = cosine_similarity(normalized_features)

Matrice normalisée :
 [[1.00000000e+00 8.07432432e-01 2.47678019e-02 3.82702703e-04
  0.00000000e+00 5.87525151e-01 0.00000000e+00 0.00000000e+00
  7.66997708e-01 1.00000000e+00 1.00000000e+00]
 [3.80587770e-01 8.64864865e-01 1.70278638e-01 0.00000000e+00
  1.00000000e+00 0.00000000e+00 7.30016167e-01 1.00000000e+00
  1.00000000e+00 7.60472797e-04 9.23351159e-01]
 [3.19392827e-01 1.00000000e+00 8.04953560e-01 5.62162162e-03
  1.42857143e-01 3.05835010e-01 8.91323873e-01 0.00000000e+00
  8.31932773e-01 2.14236051e-02 9.91087344e-01]
 [5.62545710e-02 7.36486486e-01 1.00000000e+00 1.00000000e+00
  7.14285714e-01 1.00000000e+00 8.15879289e-01 0.00000000e+00
  2.62796028e-01 7.82613419e-01 6.34581105e-01]
 [0.00000000e+00 4.59459459e-01 4.70588235e-01 0.00000000e+00
  0.00000000e+00 1.69014085e-01 6.91036465e-01 1.00000000e+00
  3.04812834e-01 0.00000000e+00 1.78253119e-01]
 [5.67172644e-03 3.20945946e-01 7.89473684e-01 2.15135135e-04
  2.85714286e-01 5.07042254e-01 6.65708640e-01 0.0000000

In [166]:
random_playlist = sp.search(q='Top 50 France', type='playlist', limit=1)
#print(random_playlist)
print("🎵 Songs from playlist 'Top 50 France' :")
# Extract the first playlist ID
"""
if random_playlist['playlists']['items']:
    public_playlist_id = random_playlist['playlists']['items'][0]['id']
    print(f"Found playlist: {random_playlist['playlists']['items'][0]['name']} (ID: {public_playlist_id})")
    print(random_playlist['playlists']['items'])
    # Fetch tracks from the playlist using playlist_items
"""
public_playlist_id = "37i9dQZEVXbMDoHDwVN2tF"  # Official Top 50 France
playlist_items = sp.playlist_items(public_playlist_id)

# Print the tracks
for idx, item in enumerate(playlist_items['items'], 1):
    track = item['track']
    if track:  # Ensure the track exists (some items might be None or local tracks)
        artists = ', '.join([artist['name'] for artist in track['artists']])
        print(f"{idx}. {track['name']} - {artists} (ID: {track['id']})")



HTTP Error for GET to https://api.spotify.com/v1/playlists/37i9dQZEVXbMDoHDwVN2tF/items with Params: {'limit': 50, 'offset': 0, 'fields': None, 'market': None, 'additional_types': 'track,episode'} returned 404 due to Resource not found


🎵 Songs from playlist 'Top 50 France' :


SpotifyException: http status: 404, code: -1 - https://api.spotify.com/v1/playlists/37i9dQZEVXbMDoHDwVN2tF/items?limit=50&offset=0&additional_types=track%2Cepisode:
 Resource not found, reason: None